# Gold Layer — Incremental Load: Facts

**Purpose**: Loads all 10 Gold fact tables from Silver layer data.

**Run order** (this notebook comes last):
```
GOLD_SETUP.ipynb  →  GOLD_INCREMENTAL_LOAD_DIMENSIONS.ipynb  →  GOLD_INCREMENTAL_LOAD_FACTS.ipynb
```

## Incremental pattern
Each fact table records the `_SILVER_LOAD_TIMESTAMP` from its source Silver records.  
On each run, only Silver rows **newer than the watermark** (`MAX(_SILVER_LOAD_TIMESTAMP)` in the Gold table) are processed and appended.

## Fact load order
1. `FACT_ORDERS` (core — all channels unified)  
2. `FACT_ORDERS_MOBILE_EXT`, `FACT_ORDERS_WHOLESALE_EXT`, `FACT_ORDERS_MARKETPLACE_EXT`  
3. `FACT_ORDER_ITEMS` (union of all item sources)  
4. `FACT_PAYMENTS`, `FACT_SHIPMENTS`, `FACT_REVIEWS`  
5. `FACT_SUPPORT_TICKETS`  
6. `FACT_USER_DAILY_ENGAGEMENT` (aggregated from APP_EVENTS)  
7. `FACT_CAMPAIGN_DAILY`

**Note:** `FACT_ORDERS` must be loaded before any other fact table (others join to it for `ORDER_KEY`).

In [ ]:
# ─────────────────────────────────────────────
# Cell 1 — Imports and Session
# ─────────────────────────────────────────────
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import (
    col, lit, current_timestamp, current_date, sql_expr,
    when, coalesce, row_number,
    max as spark_max, min as spark_min,
    count as spark_count, sum as spark_sum
)
from snowflake.snowpark import Window
from datetime import date

session = get_active_session()

GOLD_SCHEMA   = 'GOLD'
SILVER_SCHEMA = 'SILVER'
RUN_DATE      = date.today()

print(f'Session   : {session.get_current_database()}.{session.get_current_schema()}')
print(f'Run date  : {RUN_DATE}')
print(f'Gold      : {GOLD_SCHEMA}')
print(f'Silver    : {SILVER_SCHEMA}')

In [ ]:
# ─────────────────────────────────────────────
# Cell 2 — Helper Functions
# ─────────────────────────────────────────────

def get_gold_watermark(gold_table):
    """
    Returns the MAX _SILVER_LOAD_TIMESTAMP already loaded into a Gold fact table.
    Returns '1900-01-01 00:00:00' if the table is empty (first run).
    """
    try:
        result = session.sql(
            f'SELECT COALESCE(MAX(_SILVER_LOAD_TIMESTAMP)::VARCHAR, \'1900-01-01 00:00:00\') '
            f'FROM {GOLD_SCHEMA}.{gold_table}'
        ).collect()
        return result[0][0]
    except Exception:
        return '1900-01-01 00:00:00'


def get_next_key(table_name, key_column):
    """Returns the next available surrogate key integer. Returns 1 if table is empty."""
    result = session.sql(
        f'SELECT COALESCE(MAX({key_column}), 0) + 1 FROM {GOLD_SCHEMA}.{table_name}'
    ).collect()
    return int(result[0][0])


def validate(table_name):
    """Prints row count of a Gold table."""
    cnt = session.table(f'{GOLD_SCHEMA}.{table_name}').count()
    print(f'  ✓ {GOLD_SCHEMA}.{table_name}: {cnt:,} rows total')
    return cnt


EXECUTION_STATS = []
print('Helper functions ready.')

---
## FACT_ORDERS — Core Order Facts

**Sources**: WEB_ORDERS + MOB_ORDERS + WHL_ORDERS + MKT_ORDERS (aggregated to order level)  
**Note**: MKT_ORDERS is at line-item grain — aggregated here before UNION. Raw line items go to FACT_ORDER_ITEMS.

In [ ]:
# ─────────────────────────────────────────────
# Cell 3 — FACT_ORDERS: determine watermarks per channel
# Each channel is watermarked separately using the channel_key to filter
# We take ONE watermark across all channels from the Gold table
# ─────────────────────────────────────────────
wm_orders = get_gold_watermark('FACT_ORDERS')
print(f'FACT_ORDERS watermark: {wm_orders}')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 4 — FACT_ORDERS: Build unified orders CTE and insert
# ─────────────────────────────────────────────────────────────────

# ── Pre-flight: abort if critical dimensions are empty ──────────
_dim_checks = {
    'DIM_ORDER_STATUS': f'SELECT COUNT(*) FROM {GOLD_SCHEMA}.DIM_ORDER_STATUS',
    'DIM_DATE':         f'SELECT COUNT(*) FROM {GOLD_SCHEMA}.DIM_DATE',
    'DIM_CHANNEL':      f'SELECT COUNT(*) FROM {GOLD_SCHEMA}.DIM_CHANNEL',
}
_abort = False
for _dim, _sql in _dim_checks.items():
    _cnt = session.sql(_sql).collect()[0][0]
    if _cnt == 0:
        print(f'  ✗ ABORT: {_dim} is empty — run GOLD_INCREMENTAL_LOAD_DIMENSIONS first!')
        _abort = True
    else:
        print(f'  ✓ {_dim}: {_cnt:,} rows — OK')
if _abort:
    raise RuntimeError(
        'One or more critical dimension tables are empty.\n'
        'Run GOLD_INCREMENTAL_LOAD_DIMENSIONS.ipynb first, then re-run this cell.'
    )
print()

next_key = get_next_key('FACT_ORDERS', 'ORDER_KEY')

session.sql(f'''
INSERT INTO {GOLD_SCHEMA}.FACT_ORDERS (
    ORDER_KEY, ORDER_ID, CUSTOMER_KEY, ORDER_DATE_KEY, CHANNEL_KEY,
    ORDER_STATUS_KEY, ORDER_DATE, ORDER_AMOUNT, ORDER_AMOUNT_USD,
    EXCHANGE_RATE, TAX_AMOUNT, CURRENCY_CODE, PROMO_CODE, _SILVER_LOAD_TIMESTAMP
)
WITH new_orders AS (
    -- ── Web orders ──────────────────────────────────────────
    SELECT
        wo.ORDER_ID,
        wo.CUSTOMER_ID::VARCHAR       AS SOURCE_CUSTOMER_ID,
        'web'                         AS CHANNEL_ID,
        wo.ORDER_DATE,
        wo.ORDER_STATUS               AS SOURCE_STATUS,
        wo.ORDER_AMOUNT,
        NULL::DECIMAL(14,2)           AS TAX_AMOUNT,
        wo.CURRENCY_CODE,
        wo.PROMO_CODE,
        wo._SILVER_LOAD_TIMESTAMP
    FROM {SILVER_SCHEMA}.WEB_ORDERS wo
    WHERE wo._SILVER_LOAD_TIMESTAMP > '{wm_orders}'

    UNION ALL

    -- ── Mobile orders ───────────────────────────────────────
    SELECT
        mo.ORDER_ID,
        mo.CUSTOMER_ID::VARCHAR       AS SOURCE_CUSTOMER_ID,
        'mobile'                      AS CHANNEL_ID,
        mo.ORDER_DATE,
        mo.ORDER_STATUS               AS SOURCE_STATUS,
        mo.ORDER_AMOUNT,
        NULL::DECIMAL(14,2)           AS TAX_AMOUNT,
        'USD'                         AS CURRENCY_CODE,
        mo.PROMO_CODE,
        mo._SILVER_LOAD_TIMESTAMP
    FROM {SILVER_SCHEMA}.MOB_ORDERS mo
    WHERE mo._SILVER_LOAD_TIMESTAMP > '{wm_orders}'

    UNION ALL

    -- ── Wholesale orders ────────────────────────────────────
    SELECT
        wo.ORDER_ID,
        wo.CUSTOMER_ID::VARCHAR       AS SOURCE_CUSTOMER_ID,
        'wholesale'                   AS CHANNEL_ID,
        wo.ORDER_DATE,
        wo.ORDER_STATUS               AS SOURCE_STATUS,
        wo.ORDER_AMOUNT,
        wo.TAX_AMOUNT,
        'USD'                         AS CURRENCY_CODE,
        wo.PROMO_CODE,
        wo._SILVER_LOAD_TIMESTAMP
    FROM {SILVER_SCHEMA}.WHL_ORDERS wo
    WHERE wo._SILVER_LOAD_TIMESTAMP > '{wm_orders}'

    UNION ALL

    -- ── Marketplace orders (aggregated from line-item grain) ─
    SELECT
        mk.ORDER_ID,
        NULL::VARCHAR                 AS SOURCE_CUSTOMER_ID,
        'marketplace'                 AS CHANNEL_ID,
        mk.PURCHASE_DATE              AS ORDER_DATE,
        mk.ORDER_STATUS               AS SOURCE_STATUS,
        SUM(mk.LINE_TOTAL)            AS ORDER_AMOUNT,
        NULL::DECIMAL(14,2)           AS TAX_AMOUNT,
        MAX(mk.CURRENCY_CODE)         AS CURRENCY_CODE,
        MAX(mk.PROMO_CODE)            AS PROMO_CODE,
        MAX(mk._SILVER_LOAD_TIMESTAMP) AS _SILVER_LOAD_TIMESTAMP
    FROM {SILVER_SCHEMA}.MKT_ORDERS mk
    WHERE mk._SILVER_LOAD_TIMESTAMP > '{wm_orders}'
    GROUP BY mk.ORDER_ID, mk.PURCHASE_DATE, mk.ORDER_STATUS
),
keyed AS (
    SELECT
        no.*,
        IFF(no.CHANNEL_ID != 'marketplace',
            dc.CUSTOMER_KEY,
            NULL
        )                                           AS CUSTOMER_KEY,
        dd.DATE_KEY                                 AS ORDER_DATE_KEY,
        dch.CHANNEL_KEY,
        dos.ORDER_STATUS_KEY
    FROM new_orders no
    LEFT JOIN {SILVER_SCHEMA}.BRIDGE_CUSTOMER_IDENTITY bci
        ON bci.SOURCE_CUSTOMER_ID = no.SOURCE_CUSTOMER_ID
        AND bci.SOURCE_SYSTEM     = no.CHANNEL_ID
        AND bci.IS_ACTIVE         = TRUE
    LEFT JOIN {GOLD_SCHEMA}.DIM_CUSTOMER dc
        ON dc.CUSTOMER_ID   = bci.CANONICAL_CUSTOMER_ID
        AND dc.IS_CURRENT   = TRUE
    LEFT JOIN {GOLD_SCHEMA}.DIM_DATE dd
        ON dd.FULL_DATE = no.ORDER_DATE::DATE
    LEFT JOIN {GOLD_SCHEMA}.DIM_CHANNEL dch
        ON dch.CHANNEL_ID = no.CHANNEL_ID
    LEFT JOIN {GOLD_SCHEMA}.DIM_ORDER_STATUS dos
        ON dos.STATUS_CODE = no.SOURCE_STATUS
)
SELECT
    {next_key} - 1 + ROW_NUMBER() OVER (ORDER BY ORDER_ID, CHANNEL_ID)  AS ORDER_KEY,
    ORDER_ID,
    CUSTOMER_KEY,
    ORDER_DATE_KEY,
    CHANNEL_KEY,
    ORDER_STATUS_KEY,
    ORDER_DATE,
    ORDER_AMOUNT,
    ORDER_AMOUNT * 1.0              AS ORDER_AMOUNT_USD,
    1.0                             AS EXCHANGE_RATE,
    TAX_AMOUNT,
    CURRENCY_CODE,
    PROMO_CODE,
    _SILVER_LOAD_TIMESTAMP
FROM keyed k
-- ── Guard: skip orders already loaded into Gold (prevents duplicates on re-run)
WHERE NOT EXISTS (
    SELECT 1
    FROM {GOLD_SCHEMA}.FACT_ORDERS existing
    WHERE existing.ORDER_ID    = k.ORDER_ID
      AND existing.CHANNEL_KEY = k.CHANNEL_KEY
)
''').collect()

cnt = validate('FACT_ORDERS')
EXECUTION_STATS.append(('FACT_ORDERS', cnt))

---
## FACT_ORDERS Extension Tables

In [ ]:
# ─────────────────────────────────────────────
# Cell 5 — FACT_ORDERS_MOBILE_EXT
# Drive from FACT_ORDERS (mobile channel) → JOIN Silver for attributes
# NOT EXISTS guard handles incrementality — no Silver watermark needed
# ─────────────────────────────────────────────
session.sql(f'''
INSERT INTO {GOLD_SCHEMA}.FACT_ORDERS_MOBILE_EXT (ORDER_KEY, PLATFORM_TYPE, SOURCE_ORDER_ID)
SELECT
    fo.ORDER_KEY,
    mo.PLATFORM_TYPE,
    mo.SOURCE_ORDER_ID
FROM {GOLD_SCHEMA}.FACT_ORDERS fo
JOIN {GOLD_SCHEMA}.DIM_CHANNEL dch
    ON dch.CHANNEL_KEY = fo.CHANNEL_KEY
    AND LOWER(dch.CHANNEL_ID) = 'mobile'
JOIN {SILVER_SCHEMA}.MOB_ORDERS mo
    ON mo.ORDER_ID = fo.ORDER_ID
WHERE NOT EXISTS (
    SELECT 1 FROM {GOLD_SCHEMA}.FACT_ORDERS_MOBILE_EXT e
    WHERE e.ORDER_KEY = fo.ORDER_KEY
)
''').collect()

cnt = validate('FACT_ORDERS_MOBILE_EXT')
EXECUTION_STATS.append(('FACT_ORDERS_MOBILE_EXT', cnt))

In [ ]:
# ─────────────────────────────────────────────
# Cell 6 — FACT_ORDERS_WHOLESALE_EXT
# Drive from FACT_ORDERS (wholesale channel) → JOIN Silver for attributes
# ─────────────────────────────────────────────
session.sql(f'''
INSERT INTO {GOLD_SCHEMA}.FACT_ORDERS_WHOLESALE_EXT
    (ORDER_KEY, BUYER_PO_NUMBER, PAYMENT_TERMS_KEY, ORDER_AMOUNT_EXCL_TAX)
SELECT
    fo.ORDER_KEY,
    wo.BUYER_PO_NUMBER,
    dpt.PAYMENT_TERMS_KEY,
    wo.ORDER_AMOUNT_EXCL_TAX
FROM {GOLD_SCHEMA}.FACT_ORDERS fo
JOIN {GOLD_SCHEMA}.DIM_CHANNEL dch
    ON dch.CHANNEL_KEY = fo.CHANNEL_KEY
    AND LOWER(dch.CHANNEL_ID) = 'wholesale'
JOIN {SILVER_SCHEMA}.WHL_ORDERS wo
    ON wo.ORDER_ID = fo.ORDER_ID
LEFT JOIN {GOLD_SCHEMA}.DIM_PAYMENT_TERMS dpt
    ON dpt.PAYMENT_TERMS_CODE = wo.PAYMENT_TERMS
WHERE NOT EXISTS (
    SELECT 1 FROM {GOLD_SCHEMA}.FACT_ORDERS_WHOLESALE_EXT e
    WHERE e.ORDER_KEY = fo.ORDER_KEY
)
''').collect()

cnt = validate('FACT_ORDERS_WHOLESALE_EXT')
EXECUTION_STATS.append(('FACT_ORDERS_WHOLESALE_EXT', cnt))

In [ ]:
# ─────────────────────────────────────────────
# Cell 7 — FACT_ORDERS_MARKETPLACE_EXT
# Drive from FACT_ORDERS (marketplace channel) → JOIN Silver (aggregated — line-item grain)
# ─────────────────────────────────────────────
session.sql(f'''
INSERT INTO {GOLD_SCHEMA}.FACT_ORDERS_MARKETPLACE_EXT
    (ORDER_KEY, AMAZON_ORDER_ID, FULFILLMENT_CHANNEL)
SELECT
    fo.ORDER_KEY,
    MAX(mk.AMAZON_ORDER_ID)     AS AMAZON_ORDER_ID,
    MAX(mk.FULFILLMENT_CHANNEL) AS FULFILLMENT_CHANNEL
FROM {GOLD_SCHEMA}.FACT_ORDERS fo
JOIN {GOLD_SCHEMA}.DIM_CHANNEL dch
    ON dch.CHANNEL_KEY = fo.CHANNEL_KEY
    AND LOWER(dch.CHANNEL_ID) = 'marketplace'
JOIN {SILVER_SCHEMA}.MKT_ORDERS mk
    ON mk.ORDER_ID = fo.ORDER_ID
WHERE NOT EXISTS (
    SELECT 1 FROM {GOLD_SCHEMA}.FACT_ORDERS_MARKETPLACE_EXT e
    WHERE e.ORDER_KEY = fo.ORDER_KEY
)
GROUP BY fo.ORDER_KEY
''').collect()

cnt = validate('FACT_ORDERS_MARKETPLACE_EXT')
EXECUTION_STATS.append(('FACT_ORDERS_MARKETPLACE_EXT', cnt))

---
## FACT_ORDER_ITEMS — Unified Line Items

In [ ]:

# ─────────────────────────────────────────────
# Cell 8 — FACT_ORDER_ITEMS
# Sources: WEB_ORDER_ITEMS + MOB_ORDER_ITEMS + WHL_ORDER_ITEMS + MKT_ORDERS (direct, line-item grain)
# IS_RETURNED: web only | MARKETPLACE_FEE: marketplace only | Both NULL for other channels
# ─────────────────────────────────────────────
wm = get_gold_watermark('FACT_ORDER_ITEMS')
next_key = get_next_key('FACT_ORDER_ITEMS', 'ORDER_ITEM_KEY')

session.sql(f'''
INSERT INTO {GOLD_SCHEMA}.FACT_ORDER_ITEMS (
    ORDER_ITEM_KEY, ORDER_ID, ORDER_KEY, PRODUCT_KEY, CUSTOMER_KEY,
    ORDER_DATE_KEY, CHANNEL_KEY, CATEGORY_KEY,
    QUANTITY, UNIT_PRICE, LINE_TOTAL, IS_RETURNED, MARKETPLACE_FEE,
    _SILVER_LOAD_TIMESTAMP
)
WITH new_items AS (
    -- ── Web order items ──────────────────────────────────────
    SELECT
        wi.ORDER_ID,
        wi.PRODUCT_SKU,
        wi.CATEGORY_CODE,
        wi.QUANTITY,
        wi.UNIT_PRICE,
        wi.LINE_TOTAL,
        wi.IS_RETURNED,
        NULL::DECIMAL(10,2)  AS MARKETPLACE_FEE,
        'web'                AS CHANNEL_ID,
        wi._SILVER_LOAD_TIMESTAMP
    FROM {SILVER_SCHEMA}.WEB_ORDER_ITEMS wi
    WHERE wi._SILVER_LOAD_TIMESTAMP > '{wm}'

    UNION ALL

    -- ── Mobile order items ───────────────────────────────────
    SELECT
        mi.ORDER_ID,
        mi.PRODUCT_SKU,
        mi.CATEGORY_CODE,
        mi.QUANTITY,
        mi.UNIT_PRICE,
        mi.LINE_TOTAL,
        NULL::BOOLEAN        AS IS_RETURNED,
        NULL::DECIMAL(10,2)  AS MARKETPLACE_FEE,
        'mobile'             AS CHANNEL_ID,
        mi._SILVER_LOAD_TIMESTAMP
    FROM {SILVER_SCHEMA}.MOB_ORDER_ITEMS mi
    WHERE mi._SILVER_LOAD_TIMESTAMP > '{wm}'

    UNION ALL

    -- ── Wholesale order items ────────────────────────────────
    SELECT
        wli.ORDER_ID,
        wli.PRODUCT_SKU,
        wli.CATEGORY_CODE,
        wli.QUANTITY,
        wli.UNIT_PRICE,
        wli.LINE_TOTAL,
        NULL::BOOLEAN        AS IS_RETURNED,
        NULL::DECIMAL(10,2)  AS MARKETPLACE_FEE,
        'wholesale'          AS CHANNEL_ID,
        wli._SILVER_LOAD_TIMESTAMP
    FROM {SILVER_SCHEMA}.WHL_ORDER_ITEMS wli
    WHERE wli._SILVER_LOAD_TIMESTAMP > '{wm}'

    UNION ALL

    -- ── Marketplace (already line-item grain — direct pass) ──
    SELECT
        mk.ORDER_ID,
        mk.PRODUCT_SKU,
        mk.CATEGORY_CODE,
        mk.QUANTITY,
        mk.UNIT_PRICE,
        mk.LINE_TOTAL,
        NULL::BOOLEAN        AS IS_RETURNED,
        mk.MARKETPLACE_FEE,
        'marketplace'        AS CHANNEL_ID,
        mk._SILVER_LOAD_TIMESTAMP
    FROM {SILVER_SCHEMA}.MKT_ORDERS mk
    WHERE mk._SILVER_LOAD_TIMESTAMP > '{wm}'
),
keyed AS (
    SELECT
        ni.*,
        fo.ORDER_KEY,
        fo.CUSTOMER_KEY,
        fo.ORDER_DATE_KEY,
        -- Use DIM_CHANNEL key resolved from ni.CHANNEL_ID (avoids correlated subquery)
        dch.CHANNEL_KEY,
        dp.PRODUCT_KEY,
        dc.CATEGORY_KEY
    FROM new_items ni
    -- Resolve CHANNEL_KEY upfront — eliminates correlated subquery in FACT_ORDERS join
    LEFT JOIN {GOLD_SCHEMA}.DIM_CHANNEL dch
        ON dch.CHANNEL_ID = ni.CHANNEL_ID
    LEFT JOIN {GOLD_SCHEMA}.FACT_ORDERS fo
        ON fo.ORDER_ID    = ni.ORDER_ID
        AND fo.CHANNEL_KEY = dch.CHANNEL_KEY
    LEFT JOIN {GOLD_SCHEMA}.DIM_PRODUCT dp
        ON dp.PRODUCT_SKU = ni.PRODUCT_SKU AND dp.IS_CURRENT = TRUE
    LEFT JOIN {GOLD_SCHEMA}.DIM_CATEGORY dc
        ON dc.CATEGORY_CODE = ni.CATEGORY_CODE
)
SELECT
    {next_key} - 1 + ROW_NUMBER() OVER (ORDER BY ORDER_ID, CHANNEL_ID, PRODUCT_SKU)  AS ORDER_ITEM_KEY,
    ORDER_ID,
    ORDER_KEY,
    PRODUCT_KEY,
    CUSTOMER_KEY,
    ORDER_DATE_KEY,
    CHANNEL_KEY,
    CATEGORY_KEY,
    QUANTITY,
    UNIT_PRICE,
    LINE_TOTAL,
    IS_RETURNED,
    MARKETPLACE_FEE,
    _SILVER_LOAD_TIMESTAMP
FROM keyed
''').collect()

cnt = validate('FACT_ORDER_ITEMS')
EXECUTION_STATS.append(('FACT_ORDER_ITEMS', cnt))


---
## FACT_PAYMENTS

In [ ]:

# ─────────────────────────────────────────────
# Cell 9 — FACT_PAYMENTS
# Source: SILVER.PAY_TRANSACTIONS
# ─────────────────────────────────────────────
wm = get_gold_watermark('FACT_PAYMENTS')
next_key = get_next_key('FACT_PAYMENTS', 'PAYMENT_KEY')

session.sql(f'''
INSERT INTO {GOLD_SCHEMA}.FACT_PAYMENTS (
    PAYMENT_KEY, TRANSACTION_ID, ORDER_ID, ORDER_KEY,
    PAYMENT_METHOD_KEY, PAYMENT_STATUS_KEY, TRANSACTION_DATE_KEY, CHANNEL_KEY,
    PROCESSED_DATETIME, GROSS_AMOUNT, FEE_AMOUNT, NET_AMOUNT,
    CURRENCY_CODE, FAILURE_REASON_CODE, IS_FIRST_TRANSACTION, GATEWAY_REGION,
    _SILVER_LOAD_TIMESTAMP
)
SELECT
    {next_key} - 1 + ROW_NUMBER() OVER (ORDER BY pt.TRANSACTION_ID)  AS PAYMENT_KEY,
    pt.TRANSACTION_ID,
    pt.ORDER_ID,
    fo.ORDER_KEY,
    dpm.PAYMENT_METHOD_KEY,
    dps.PAYMENT_STATUS_KEY,
    dd.DATE_KEY    AS TRANSACTION_DATE_KEY,
    dch.CHANNEL_KEY,
    -- Cast TIMESTAMP_TZ → TIMESTAMP_NTZ to match Gold table DDL
    pt.PROCESSED_DATETIME::TIMESTAMP_NTZ  AS PROCESSED_DATETIME,
    pt.GROSS_AMOUNT,
    pt.FEE_AMOUNT,
    pt.NET_AMOUNT,
    pt.CURRENCY_CODE,
    pt.FAILURE_REASON_CODE,
    pt.IS_FIRST_TRANSACTION,
    pt.GATEWAY_REGION,
    pt._SILVER_LOAD_TIMESTAMP
FROM {SILVER_SCHEMA}.PAY_TRANSACTIONS pt
-- Resolve ORDER_KEY (match on ORDER_ID; Web/Mobile only — wholesale & marketplace use invoices/Amazon Pay)
LEFT JOIN {GOLD_SCHEMA}.FACT_ORDERS fo
    ON fo.ORDER_ID     = pt.ORDER_ID
    AND fo.CHANNEL_KEY IN (
        SELECT CHANNEL_KEY FROM {GOLD_SCHEMA}.DIM_CHANNEL
        WHERE CHANNEL_ID IN ('web', 'mobile')
    )
LEFT JOIN {GOLD_SCHEMA}.DIM_PAYMENT_METHOD dpm
    ON dpm.PAYMENT_METHOD_CODE = pt.PAYMENT_METHOD_CODE
LEFT JOIN {GOLD_SCHEMA}.DIM_PAYMENT_STATUS dps
    ON dps.STATUS_CODE = pt.PAYMENT_STATUS_CODE
LEFT JOIN {GOLD_SCHEMA}.DIM_DATE dd
    ON dd.FULL_DATE = pt.PROCESSED_DATETIME::DATE
-- FIX: PAY_TRANSACTIONS.SOURCE_CHANNEL uses abbreviations ('WEB', 'MOB')
--      but DIM_CHANNEL.CHANNEL_ID uses full names ('web', 'mobile').
--      Map abbreviations to full CHANNEL_ID values before joining.
LEFT JOIN {GOLD_SCHEMA}.DIM_CHANNEL dch
    ON dch.CHANNEL_ID = CASE
        WHEN UPPER(pt.SOURCE_CHANNEL) = 'WEB' THEN 'web'
        WHEN UPPER(pt.SOURCE_CHANNEL) = 'MOB' THEN 'mobile'
        ELSE LOWER(pt.SOURCE_CHANNEL)
    END
WHERE pt._SILVER_LOAD_TIMESTAMP > '{wm}'
  -- Guard: skip transactions already loaded into Gold (prevents duplicates on re-run)
  AND NOT EXISTS (
      SELECT 1
      FROM {GOLD_SCHEMA}.FACT_PAYMENTS existing
      WHERE existing.TRANSACTION_ID = pt.TRANSACTION_ID
  )
''').collect()

cnt = validate('FACT_PAYMENTS')
EXECUTION_STATS.append(('FACT_PAYMENTS', cnt))


---
## FACT_SHIPMENTS

In [ ]:
# ─────────────────────────────────────────────
# Cell 10 — FACT_SHIPMENTS
# Source: SILVER.SHP_SHIPMENTS
# Derived: DAYS_TO_DELIVER, ON_TIME_DELIVERY_FLAG (baseline: ≤ 7 business days)
# ─────────────────────────────────────────────
wm = get_gold_watermark('FACT_SHIPMENTS')
next_key = get_next_key('FACT_SHIPMENTS', 'SHIPMENT_KEY')

session.sql(f'''
INSERT INTO {GOLD_SCHEMA}.FACT_SHIPMENTS (
    SHIPMENT_KEY, SHIPMENT_ID, ORDER_ID, ORDER_KEY,
    SHIPMENT_DATE_KEY, DELIVERY_DATE_KEY, CHANNEL_KEY,
    ORIGIN_FACILITY, PICKUP_DATETIME, DELIVERY_DATETIME,
    PACKAGE_WEIGHT_KG, WEIGHT_UNIT_ORIGINAL, SHIPMENT_STATUS,
    SIGNATURE_REQUIRED, INSURANCE_VALUE,
    DAYS_TO_DELIVER, ON_TIME_DELIVERY_FLAG,
    _SILVER_LOAD_TIMESTAMP
)
SELECT
    {next_key} - 1 + ROW_NUMBER() OVER (ORDER BY sh.SHIPMENT_ID)  AS SHIPMENT_KEY,
    sh.SHIPMENT_ID,
    sh.ORDER_ID,
    fo.ORDER_KEY,
    dd_pickup.DATE_KEY                        AS SHIPMENT_DATE_KEY,
    dd_deliver.DATE_KEY                       AS DELIVERY_DATE_KEY,
    dch.CHANNEL_KEY,
    sh.ORIGIN_FACILITY,
    sh.PICKUP_DATETIME,
    sh.DELIVERY_DATETIME,
    sh.PACKAGE_WEIGHT_KG,
    sh.WEIGHT_UNIT_ORIGINAL,
    sh.SHIPMENT_STATUS,
    sh.SIGNATURE_REQUIRED,
    sh.INSURANCE_VALUE,
    -- DAYS_TO_DELIVER: NULL for in-transit shipments (DELIVERY_DATETIME is NULL)
    DATEDIFF(day, sh.PICKUP_DATETIME, sh.DELIVERY_DATETIME)  AS DAYS_TO_DELIVER,
    -- ON_TIME: delivered within 7 days
    IFF(
        sh.DELIVERY_DATETIME IS NOT NULL
        AND DATEDIFF(day, sh.PICKUP_DATETIME, sh.DELIVERY_DATETIME) <= 7,
        TRUE, FALSE
    )                                                        AS ON_TIME_DELIVERY_FLAG,
    sh._SILVER_LOAD_TIMESTAMP
FROM {SILVER_SCHEMA}.SHP_SHIPMENTS sh
LEFT JOIN {GOLD_SCHEMA}.FACT_ORDERS fo
    ON fo.ORDER_ID = sh.ORDER_ID
LEFT JOIN {GOLD_SCHEMA}.DIM_DATE dd_pickup
    ON dd_pickup.FULL_DATE = sh.PICKUP_DATETIME::DATE
LEFT JOIN {GOLD_SCHEMA}.DIM_DATE dd_deliver
    ON dd_deliver.FULL_DATE = sh.DELIVERY_DATETIME::DATE
LEFT JOIN {GOLD_SCHEMA}.DIM_CHANNEL dch
    ON LOWER(dch.CHANNEL_ID) = LOWER(sh.ORDER_SOURCE_SYSTEM)
WHERE sh._SILVER_LOAD_TIMESTAMP > '{wm}'
''').collect()

cnt = validate('FACT_SHIPMENTS')
EXECUTION_STATS.append(('FACT_SHIPMENTS', cnt))

---
## FACT_REVIEWS

In [ ]:
# ─────────────────────────────────────────────
# Cell 11 — FACT_REVIEWS
# Source: SILVER.REV_REVIEWS
# Cannot link to CUSTOMER_KEY or ORDER_KEY (not available in reviews data)
# ─────────────────────────────────────────────
wm = get_gold_watermark('FACT_REVIEWS')
next_key = get_next_key('FACT_REVIEWS', 'REVIEW_KEY')

session.sql(f'''
INSERT INTO {GOLD_SCHEMA}.FACT_REVIEWS (
    REVIEW_KEY, REVIEW_ID, PRODUCT_KEY, CATEGORY_KEY, REVIEW_DATE_KEY,
    REVIEWER_HANDLE, STAR_RATING, VERIFIED_PURCHASE,
    VERIFIED_SOURCE, MODERATION_STATUS, HELPFUL_VOTES,
    _SILVER_LOAD_TIMESTAMP
)
SELECT
    {next_key} - 1 + ROW_NUMBER() OVER (ORDER BY rr.REVIEW_ID)  AS REVIEW_KEY,
    rr.REVIEW_ID,
    dp.PRODUCT_KEY,
    dc.CATEGORY_KEY,
    dd.DATE_KEY   AS REVIEW_DATE_KEY,
    rr.REVIEWER_HANDLE,
    rr.STAR_RATING,
    rr.VERIFIED_PURCHASE,
    rr.VERIFIED_SOURCE,
    rr.MODERATION_STATUS,
    rr.HELPFUL_VOTES,
    rr._SILVER_LOAD_TIMESTAMP
FROM {SILVER_SCHEMA}.REV_REVIEWS rr
LEFT JOIN {GOLD_SCHEMA}.DIM_PRODUCT dp
    ON dp.PRODUCT_SKU = rr.PRODUCT_SKU AND dp.IS_CURRENT = TRUE
LEFT JOIN {GOLD_SCHEMA}.DIM_CATEGORY dc
    ON dc.CATEGORY_CODE = rr.CATEGORY_CODE
LEFT JOIN {GOLD_SCHEMA}.DIM_DATE dd
    ON dd.FULL_DATE = rr.REVIEW_DATE
WHERE rr._SILVER_LOAD_TIMESTAMP > '{wm}'
''').collect()

cnt = validate('FACT_REVIEWS')
EXECUTION_STATS.append(('FACT_REVIEWS', cnt))

---
## FACT_SUPPORT_TICKETS

In [ ]:

# ─────────────────────────────────────────────
# Cell 12 — FACT_SUPPORT_TICKETS
# Source: SILVER.SUP_TICKETS
# CUSTOMER_KEY: resolved via REQUESTER_EMAIL → DIM_CUSTOMER (IS_CURRENT = TRUE)
# REQUESTER_EMAIL and REQUESTER_NAME are NOT stored in the fact (PII deduplication)
# CSAT_SCORE: Silver stores text ('good'/'neutral'/'bad') → mapped to numeric 5/3/1
# ─────────────────────────────────────────────
wm = get_gold_watermark('FACT_SUPPORT_TICKETS')
next_key = get_next_key('FACT_SUPPORT_TICKETS', 'TICKET_KEY')

session.sql(f'''
INSERT INTO {GOLD_SCHEMA}.FACT_SUPPORT_TICKETS (
    TICKET_KEY, TICKET_ID, ORDER_ID, ORDER_KEY, CUSTOMER_KEY,
    CREATED_DATE_KEY, SOLVED_DATE_KEY, CHANNEL_KEY,
    ASSIGNEE, TAGS, TICKET_STATUS,
    CREATED_DATETIME, SOLVED_DATETIME, FIRST_REPLY_HOURS, CSAT_SCORE,
    DAYS_TO_SOLVE, IS_SOLVED,
    _SILVER_LOAD_TIMESTAMP
)
SELECT
    {next_key} - 1 + ROW_NUMBER() OVER (ORDER BY st.TICKET_ID)  AS TICKET_KEY,
    st.TICKET_ID,
    st.ORDER_ID,
    fo.ORDER_KEY,
    -- CUSTOMER_KEY via requester email → DIM_CUSTOMER (current record)
    dc.CUSTOMER_KEY,
    dd_created.DATE_KEY   AS CREATED_DATE_KEY,
    dd_solved.DATE_KEY    AS SOLVED_DATE_KEY,
    dch.CHANNEL_KEY,
    st.ASSIGNEE,
    st.TAGS,
    st.TICKET_STATUS,
    st.CREATED_DATETIME,
    st.SOLVED_DATETIME,
    st.FIRST_REPLY_HOURS,
    -- Map text CSAT labels → numeric (5=good, 3=neutral, 1=bad); fall back to TRY_TO_DECIMAL for numeric strings
    CASE LOWER(TRIM(st.CSAT_SCORE))
        WHEN 'good'    THEN 5
        WHEN 'neutral' THEN 3
        WHEN 'bad'     THEN 1
        ELSE TRY_TO_DECIMAL(st.CSAT_SCORE, 5, 2)
    END                                                         AS CSAT_SCORE,
    -- DAYS_TO_SOLVE: NULL for open tickets
    DATEDIFF(day, st.CREATED_DATETIME, st.SOLVED_DATETIME)      AS DAYS_TO_SOLVE,
    IFF(LOWER(st.TICKET_STATUS) = 'solved', TRUE, FALSE)        AS IS_SOLVED,
    st._SILVER_LOAD_TIMESTAMP
FROM {SILVER_SCHEMA}.SUP_TICKETS st
-- Link to FACT_ORDERS via ORDER_ID
LEFT JOIN {GOLD_SCHEMA}.FACT_ORDERS fo
    ON fo.ORDER_ID = st.ORDER_ID
-- Resolve CUSTOMER_KEY via email
LEFT JOIN {GOLD_SCHEMA}.DIM_CUSTOMER dc
    ON LOWER(dc.EMAIL_ADDRESS) = LOWER(st.REQUESTER_EMAIL)
    AND dc.IS_CURRENT = TRUE
LEFT JOIN {GOLD_SCHEMA}.DIM_DATE dd_created
    ON dd_created.FULL_DATE = st.CREATED_DATETIME::DATE
LEFT JOIN {GOLD_SCHEMA}.DIM_DATE dd_solved
    ON dd_solved.FULL_DATE = st.SOLVED_DATETIME::DATE
LEFT JOIN {GOLD_SCHEMA}.DIM_CHANNEL dch
    ON LOWER(dch.CHANNEL_ID) = LOWER(st.ORDER_SOURCE)
WHERE st._SILVER_LOAD_TIMESTAMP > '{wm}'
''').collect()

cnt = validate('FACT_SUPPORT_TICKETS')
EXECUTION_STATS.append(('FACT_SUPPORT_TICKETS', cnt))


---
## FACT_USER_DAILY_ENGAGEMENT

Aggregated from `APP_EVENTS` at **daily grain per user**. Raw events stay in Silver.  
Incremental key: if a USER_ID + DATE combination already exists in Gold, it is skipped.

In [ ]:

# ─────────────────────────────────────────────
# Cell 13 — FACT_USER_DAILY_ENGAGEMENT
# Source: SILVER.APP_EVENTS (aggregated daily per USER_ID)
# CUSTOMER_KEY: USER_ID → MOB_CUSTOMERS → BRIDGE_CUSTOMER_IDENTITY → DIM_CUSTOMER
# NULL for ~15% guest sessions (no USER_ID)
# APP_VERSION: latest version used that day — resolved in separate CTE to avoid
#              mixing window functions with GROUP BY aggregates (Snowflake error 000979)
# ─────────────────────────────────────────────
wm = get_gold_watermark('FACT_USER_DAILY_ENGAGEMENT')
next_key = get_next_key('FACT_USER_DAILY_ENGAGEMENT', 'USER_DAILY_KEY')

session.sql(f'''
INSERT INTO {GOLD_SCHEMA}.FACT_USER_DAILY_ENGAGEMENT (
    USER_DAILY_KEY, USER_ID, CUSTOMER_KEY, ACTIVITY_DATE_KEY, ACTIVITY_DATE,
    APP_VERSION, IS_AUTHENTICATED, IS_VPN_DETECTED,
    SESSION_COUNT, TOTAL_EVENTS, FIRST_EVENT_TIME, LAST_EVENT_TIME, ACTIVE_MINUTES,
    LOGIN_COUNT, PRODUCT_VIEW_COUNT, CATEGORY_BROWSE_COUNT, SEARCH_COUNT,
    ADD_TO_CART_COUNT, CHECKOUT_START_COUNT, CHECKOUT_COMPLETE_COUNT, WISHLIST_ADD_COUNT,
    UNIQUE_PRODUCTS_VIEWED, UNIQUE_CATEGORIES_BROWSED,
    _SILVER_LOAD_TIMESTAMP
)
WITH new_events AS (
    SELECT *
    FROM {SILVER_SCHEMA}.APP_EVENTS
    WHERE _SILVER_LOAD_TIMESTAMP > '{wm}'
      AND USER_ID IS NOT NULL  -- exclude unauthenticated guest sessions
),
-- Step 1: isolate latest APP_VERSION per user/day using QUALIFY
--         (cannot mix LAST_VALUE window function with GROUP BY in same SELECT)
latest_app_version AS (
    SELECT
        USER_ID,
        EVENT_TIMESTAMP::DATE  AS ACTIVITY_DATE,
        APP_VERSION
    FROM new_events
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY USER_ID, EVENT_TIMESTAMP::DATE
        ORDER BY EVENT_TIMESTAMP DESC
    ) = 1
),
-- Step 2: all aggregate measures — pure GROUP BY, no window functions
daily_agg AS (
    SELECT
        USER_ID,
        EVENT_TIMESTAMP::DATE                              AS ACTIVITY_DATE,
        MAX(IS_AUTHENTICATED::INTEGER)::BOOLEAN            AS IS_AUTHENTICATED,
        MAX(IS_VPN_DETECTED::INTEGER)::BOOLEAN             AS IS_VPN_DETECTED,
        COUNT(DISTINCT SESSION_ID)                         AS SESSION_COUNT,
        COUNT(*)                                           AS TOTAL_EVENTS,
        MIN(EVENT_TIMESTAMP)                               AS FIRST_EVENT_TIME,
        MAX(EVENT_TIMESTAMP)                               AS LAST_EVENT_TIME,
        DATEDIFF(minute, MIN(EVENT_TIMESTAMP), MAX(EVENT_TIMESTAMP))
                                                           AS ACTIVE_MINUTES,
        SUM(IFF(EVENT_TYPE = 'account_login',      1, 0))  AS LOGIN_COUNT,
        SUM(IFF(EVENT_TYPE = 'product_view',        1, 0))  AS PRODUCT_VIEW_COUNT,
        SUM(IFF(EVENT_TYPE = 'category_browse',     1, 0))  AS CATEGORY_BROWSE_COUNT,
        SUM(IFF(EVENT_TYPE = 'search',              1, 0))  AS SEARCH_COUNT,
        SUM(IFF(EVENT_TYPE = 'add_to_cart',         1, 0))  AS ADD_TO_CART_COUNT,
        SUM(IFF(EVENT_TYPE = 'checkout_start',      1, 0))  AS CHECKOUT_START_COUNT,
        SUM(IFF(EVENT_TYPE = 'checkout_complete',   1, 0))  AS CHECKOUT_COMPLETE_COUNT,
        SUM(IFF(EVENT_TYPE = 'wishlist_add',        1, 0))  AS WISHLIST_ADD_COUNT,
        COUNT(DISTINCT PRODUCT_SKU)                        AS UNIQUE_PRODUCTS_VIEWED,
        COUNT(DISTINCT CATEGORY_CODE)                      AS UNIQUE_CATEGORIES_BROWSED,
        MAX(_SILVER_LOAD_TIMESTAMP)                        AS _SILVER_LOAD_TIMESTAMP
    FROM new_events
    GROUP BY USER_ID, EVENT_TIMESTAMP::DATE
)
SELECT
    {next_key} - 1 + ROW_NUMBER() OVER (ORDER BY da.USER_ID, da.ACTIVITY_DATE)  AS USER_DAILY_KEY,
    da.USER_ID,
    dc.CUSTOMER_KEY,
    dd.DATE_KEY    AS ACTIVITY_DATE_KEY,
    da.ACTIVITY_DATE,
    -- APP_VERSION from separate CTE (avoids window+GROUP BY conflict)
    lav.APP_VERSION,
    da.IS_AUTHENTICATED,
    da.IS_VPN_DETECTED,
    da.SESSION_COUNT,
    da.TOTAL_EVENTS,
    da.FIRST_EVENT_TIME,
    da.LAST_EVENT_TIME,
    da.ACTIVE_MINUTES,
    da.LOGIN_COUNT,
    da.PRODUCT_VIEW_COUNT,
    da.CATEGORY_BROWSE_COUNT,
    da.SEARCH_COUNT,
    da.ADD_TO_CART_COUNT,
    da.CHECKOUT_START_COUNT,
    da.CHECKOUT_COMPLETE_COUNT,
    da.WISHLIST_ADD_COUNT,
    da.UNIQUE_PRODUCTS_VIEWED,
    da.UNIQUE_CATEGORIES_BROWSED,
    da._SILVER_LOAD_TIMESTAMP
FROM daily_agg da
-- Join latest app version
LEFT JOIN latest_app_version lav
    ON lav.USER_ID       = da.USER_ID
    AND lav.ACTIVITY_DATE = da.ACTIVITY_DATE
-- CUSTOMER_KEY: USER_ID = MOB_CUSTOMERS.CUSTOMER_ID → BRIDGE → DIM_CUSTOMER
LEFT JOIN {SILVER_SCHEMA}.MOB_CUSTOMERS mc
    ON mc.CUSTOMER_ID = da.USER_ID
LEFT JOIN {SILVER_SCHEMA}.BRIDGE_CUSTOMER_IDENTITY bci
    ON bci.SOURCE_CUSTOMER_ID = mc.CUSTOMER_ID
    AND bci.SOURCE_SYSTEM     = 'mobile'
    AND bci.IS_ACTIVE         = TRUE
LEFT JOIN {GOLD_SCHEMA}.DIM_CUSTOMER dc
    ON dc.CUSTOMER_ID = bci.CANONICAL_CUSTOMER_ID
    AND dc.IS_CURRENT = TRUE
LEFT JOIN {GOLD_SCHEMA}.DIM_DATE dd
    ON dd.FULL_DATE = da.ACTIVITY_DATE
-- Skip USER_ID + DATE pairs already loaded
WHERE NOT EXISTS (
    SELECT 1 FROM {GOLD_SCHEMA}.FACT_USER_DAILY_ENGAGEMENT g
    WHERE g.USER_ID       = da.USER_ID
      AND g.ACTIVITY_DATE = da.ACTIVITY_DATE
)
''').collect()

cnt = validate('FACT_USER_DAILY_ENGAGEMENT')
EXECUTION_STATS.append(('FACT_USER_DAILY_ENGAGEMENT', cnt))


---
## FACT_CAMPAIGN_DAILY

In [ ]:
# ─────────────────────────────────────────────
# Cell 14 — FACT_CAMPAIGN_DAILY
# Source: SILVER.MKTG_CAMPAIGNS (already at daily grain)
# Campaign attributes live in DIM_CAMPAIGN; only measures go here
# MARKETING_CHANNEL_KEY resolved from DIM_MARKETING_CHANNEL (NOT DIM_CHANNEL)
# ─────────────────────────────────────────────
wm = get_gold_watermark('FACT_CAMPAIGN_DAILY')
next_key = get_next_key('FACT_CAMPAIGN_DAILY', 'CAMPAIGN_DAILY_KEY')

session.sql(f'''
INSERT INTO {GOLD_SCHEMA}.FACT_CAMPAIGN_DAILY (
    CAMPAIGN_DAILY_KEY, CAMPAIGN_KEY, DATE_KEY, MARKETING_CHANNEL_KEY,
    DATE, AMOUNT_SPENT, IMPRESSIONS, CLICKS, CONVERSIONS,
    CTR_PERCENT, CPC_AMOUNT, CPA_AMOUNT,
    _SILVER_LOAD_TIMESTAMP
)
SELECT
    {next_key} - 1 + ROW_NUMBER() OVER (ORDER BY mc.CAMPAIGN_ID, mc.DATE)  AS CAMPAIGN_DAILY_KEY,
    dc.CAMPAIGN_KEY,
    dd.DATE_KEY,
    dmc.MARKETING_CHANNEL_KEY,
    mc.DATE,
    mc.AMOUNT_SPENT,
    mc.IMPRESSIONS,
    mc.CLICKS,
    mc.CONVERSIONS,
    mc.CTR_PERCENT,
    mc.CPC_AMOUNT,
    mc.CPA_AMOUNT,
    mc._SILVER_LOAD_TIMESTAMP
FROM {SILVER_SCHEMA}.MKTG_CAMPAIGNS mc
LEFT JOIN {GOLD_SCHEMA}.DIM_CAMPAIGN dc
    ON dc.CAMPAIGN_ID = mc.CAMPAIGN_ID AND dc.IS_CURRENT = TRUE
LEFT JOIN {GOLD_SCHEMA}.DIM_DATE dd
    ON dd.FULL_DATE = mc.DATE
-- MARKETING_CHANNEL_KEY: maps email/social/search/display → DIM_MARKETING_CHANNEL
LEFT JOIN {GOLD_SCHEMA}.DIM_MARKETING_CHANNEL dmc
    ON LOWER(dmc.MARKETING_CHANNEL_ID) = LOWER(mc.CHANNEL)
WHERE mc._SILVER_LOAD_TIMESTAMP > '{wm}'
''').collect()

cnt = validate('FACT_CAMPAIGN_DAILY')
EXECUTION_STATS.append(('FACT_CAMPAIGN_DAILY', cnt))

---
## Final Run Summary

In [ ]:
# ─────────────────────────────────────────────
# Cell 15 — Fact Load Run Summary
# ─────────────────────────────────────────────
print('='*60)
print(f'  Gold Fact Load Summary — {RUN_DATE}')
print('='*60)
print(f'  {"Fact Table":<40} {"Total Rows":>10}')
print('-'*60)
for table, rows in EXECUTION_STATS:
    print(f'  {table:<40} {rows:>10,}')
print('='*60)

errors = [t for t, r in EXECUTION_STATS if r == 0]
if errors:
    print(f'  [!] Tables with 0 rows: {errors}')
    print('  Check watermark values and Silver table row counts.')
else:
    print('  ✓ All fact tables loaded successfully.')
    print('  ✓ Gold layer is ready for analytics queries.')
print('='*60)